In [ ]:
!mkdir -p /content/dataset
!curl -L -o /content/dataset/data.txt https://huggingface.co/datasets/ai4bharat/IndicCorpV2/resolve/main/data/hi-1.txt?download=true

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   913  100   913    0     0   5737      0 --:--:-- --:--:-- --:--:--  5778
100 24.8G  100 24.8G    0     0  67.1M      0  0:06:18  0:06:18 --:--:-- 81.6M


In [12]:
import os

os.environ['KAGGLE_USERNAME'] = "devsvnit"
os.environ['KAGGLE_KEY'] = "KGAT_ead91e9f0b9c365710cbd699a8f591f3"
os.environ["KAGGLE_API_TOKEN"] = "KGAT_ead91e9f0b9c365710cbd699a8f591f3"
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
  f.write("KGAT_ead91e9f0b9c365710cbd699a8f591f3")

!pip install -q kaggle

In [ ]:
import re
import os
import json
from kaggle.api.kaggle_api_extended import KaggleApi
import tqdm
import pyarrow as pa
import pyarrow.parquet as pq
class Tokenizer:
  def __init__(self, data_path= "/content/dataset", dataset_slug="hindi-indiccorp-tokenized"):
    self.dataset_slug= dataset_slug
    self.url = r"(?:https?://)?(?:www\.)?[-a-zA-Z0-9@:%._\+~#=]{1,256}\.[a-z]{2,6}\b(?:[-a-zA-Z0-9@:%_\+.~#?&//=]*)"
    self.number= r"[+-]?[\d\u0966-\u096F]+(?:[\.\-,][\d\u0966-\u096F]+)*"
    self.email= r"[\w\-\.]+@(?:[\w-]+\.)+[\w-]{2,4}"
    self.date= r"[\d\u0966-\u096F]{1,2}[/-][\d\u0966-\u096F]{1,2}[/-][\d\u0966-\u096F]{2,4}"
    self.punctuations= r"[\u0964\u0965\u2000-\u206F.?;,:'\\\/\"!#%&*\(\)+\-_`=\[\]{}^@~|<>]"
    self.word= r"[\u0900-\u097F\u200C\u200D\w]+"

    self.sentence= r"(?<=[\u0964\u0965!?.])\s+|\n+"

    self.data_path= data_path
    self.input_file = os.path.join(self.data_path, "data.txt")

    self.word_exp= re.compile("|".join([self.email,
                                        self.url,
                                        self.date,
                                        self.number,
                                        self.punctuations,
                                        self.word]))

  def tokenize_sentence(self):
    with open(os.path.join(self.data_path, "data.txt"), 'r', encoding='utf-8') as f:
      self.data= f.read()
    self.tokenized_sentences= [s.strip() for s in re.split(self.sentence, self.data) if s.strip()]
    with open(os.path.join(self.data_path, "sentence.txt"), 'w', encoding='utf-8') as f:
      f.write('\n'.join(self.tokenized_sentences))
    with open(os.path.join(self.data_path, "stats.txt"), 'a', encoding='utf-8') as f:
      f.write(f"Number of sentences: {len(self.tokenized_sentences)}\n")
      return

  def tokenize_word(self):
    num_words= 0
    with open(os.path.join(self.data_path, "words.txt"), 'w', encoding='utf-8') as f:
      for sentence in self.tokenized_sentences:
        tokenized_words= self.word_exp.findall(sentence)
        f.write('\t'.join(tokenized_words))
        f.write("\n")
        num_words+=len(tokenized_words)
    with open(os.path.join(self.data_path, "stats.txt"), 'a', encoding='utf-8') as f:
      f.write(f"Number of words: {num_words}\n")
    return


  def run(self):
    self.tokenize_sentence()
    self.tokenize_word()




  def process_and_upload(self, batch_size=100000):
          num_sentences = 0
          num_words = 0

          sent_parquet_path = os.path.join(self.data_path, "sentences.parquet")
          words_parquet_path = os.path.join(self.data_path, "words.parquet")

          sent_schema = pa.schema([('sentence', pa.string())])
          words_schema = pa.schema([('tokens', pa.list_(pa.string()))])

          sent_writer = pq.ParquetWriter(sent_parquet_path, sent_schema, compression='snappy')
          words_writer = pq.ParquetWriter(words_parquet_path, words_schema, compression='snappy')

          batch_sentences = []
          batch_words = []

          print("Starting batch-stream tokenization to Parquet format...")
          with open(self.input_file, 'r', encoding='utf-8') as infile:
              for line in infile:
                  if not line.strip():
                      continue

                  sentences = [s.strip() for s in re.split(self.sentence, line) if s.strip()]
                  for s in sentences:
                      words = self.word_exp.findall(s)

                      batch_sentences.append(s)
                      batch_words.append(words)

                      num_sentences += 1
                      num_words += len(words)

                      if len(batch_sentences) >= batch_size:
                          sent_table = pa.Table.from_arrays([pa.array(batch_sentences)], schema=sent_schema)
                          sent_writer.write_table(sent_table)

                          words_table = pa.Table.from_arrays([pa.array(batch_words)], schema=words_schema)
                          words_writer.write_table(words_table)

                          batch_sentences.clear()
                          batch_words.clear()

          if batch_sentences:
              sent_table = pa.Table.from_arrays([pa.array(batch_sentences)], schema=sent_schema)
              sent_writer.write_table(sent_table)

              words_table = pa.Table.from_arrays([pa.array(batch_words)], schema=words_schema)
              words_writer.write_table(words_table)

          sent_writer.close()
          words_writer.close()

          with open(os.path.join(self.data_path, "stats.txt"), 'w', encoding='utf-8') as f:
              f.write(f"Number of sentences: {num_sentences}\n")
              f.write(f"Number of words: {num_words}\n")

          print(f"Tokenization complete! Sentences: {num_sentences}, Words: {num_words}")

          if os.path.exists(self.input_file):
              os.remove(self.input_file)
              print(f"Deleted raw file '{self.input_file}' to free disk space.")

          self._upload_to_kaggle()

  def _upload_to_kaggle(self):
      username = os.environ.get('KAGGLE_USERNAME')
      dataset_id = f"{username}/{self.dataset_slug}"

      metadata = {
          "title": "hi-1 IndicCorpV2 Tokenized Parquet",
          "id": dataset_id,
          "licenses": [{"name": "CC0-1.0"}]
      }

      metadata_path = os.path.join(self.data_path, "dataset-metadata.json")
      with open(metadata_path, 'w', encoding='utf-8') as f:
          json.dump(metadata, f, indent=4)

      print(f"Uploading Parquet dataset to Kaggle ({dataset_id})...")
      api = KaggleApi()
      api.authenticate()
      for txt_file in ["sentence.txt", "words.txt"]:
        path = os.path.join(self.data_path, txt_file)
        if os.path.exists(path):
            os.remove(path)
      api.dataset_create_new(folder=self.data_path, public=False, quiet=False)
      print("Parquet dataset uploaded to Kaggle successfully!")

In [ ]:
# tokenizer= Tokenizer()
# tokenizer.run()
tokenizer = Tokenizer(dataset_slug="hindi-indiccorp-tokenized")
tokenizer.process_and_upload()

Starting batch-stream tokenization to Parquet format...
Tokenization complete! Sentences: 115522757, Words: 2158502996
Deleted raw file '/content/dataset/data.txt' to free disk space.
Uploading Parquet dataset to Kaggle (devsvnit/hindi-indiccorp-tokenized)...
Starting upload for file stats.txt


HTTPError: 401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/blobs.BlobApiService/StartBlobUpload

In [8]:
tokenizer = Tokenizer(dataset_slug="hindi-indiccorp-tokenized")
tokenizer._upload_to_kaggle()

Uploading Parquet dataset to Kaggle (devsvnit/hindi-indiccorp-tokenized)...
Starting upload for file stats.txt
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'


100%|██████████| 59.0/59.0 [00:00<00:00, 143B/s]


Upload successful: stats.txt (59B)
Starting upload for file words.parquet
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'


100%|██████████| 9.20G/9.20G [02:12<00:00, 74.6MB/s]


Upload successful: words.parquet (9GB)
Starting upload for file sentences.parquet


100%|██████████| 10.4G/10.4G [01:42<00:00, 109MB/s]


Upload successful: sentences.parquet (10GB)
Parquet dataset uploaded to Kaggle successfully!


In [11]:
import pandas as pd
import pyarrow.dataset as ds
dataset = ds.dataset("/content/dataset/words.parquet", format="parquet")
df = dataset.head(100).to_pandas()

df

,tokens
0,"[लोगों, को, बिलों, संबंधी, सुविधा, देना, ही, उ..."
1,"[इनेलो, 1987, में, उस, वक्त, ऐसे, ही, दोराहे, ..."
2,"[हालांकि, तब, पार्टी, पर, देवीलाल, की, मजबूत, ..."
3,"[1989, में, देवीलाल, केन्द्र, की, राजनीति, में..."
4,"[उन, परिस्थितियों, में, देवीलाल, ने, कड़ा, निर..."
...,...
95,"[आदिवासियों, ने, यह, भी, कहा, कि, सबरीमाला, मं..."
96,"[45, सीट, जीत, पाएंगे, ?]"
97,"[हाउसफुल, 3, ', हाउसफुल, ', सीरिज, की, सबसे, क..."
98,"[हंसने, के, लिए, इससे, बेहतर, कई, विकल्प, मौजू..."


In [13]:
dataset = ds.dataset("/content/dataset/sentences.parquet", format="parquet")
df = dataset.head(100).to_pandas()

df

,sentence
0,लोगों को बिलों संबंधी सुविधा देना ही उनका काम
1,इनेलो 1987 में उस वक्त ऐसे ही दोराहे पर खड़ी थ...
2,हालांकि तब पार्टी पर देवीलाल की मजबूत पकड़ के ...
3,1989 में देवीलाल केन्द्र की राजनीति में सक्रिय...
4,उन परिस्थितियों में देवीलाल ने कड़ा निर्णय लेत...
...,...
95,आदिवासियों ने यह भी कहा कि सबरीमाला मंदिर और इ...
96,45 सीट जीत पाएंगे?
97,हाउसफुल 3 'हाउसफुल' सीरिज की सबसे कमजोर फिल्म है।
98,हंसने के लिए इससे बेहतर कई विकल्प मौजूद हैं।


In [14]:
with open("/content/dataset/stats.txt", 'r') as f:
  print(f.read())

Number of sentences: 115522757
Number of words: 2158502996

